# Riemannian Embedded Transformer

In [9]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
import time

class PoincareManifold:
    def __init__(self, c=1.0):
        self.c = c

    def mobius_add(self, x, y):
        x2, y2 = torch.sum(x*x, -1, True), torch.sum(y*y, -1, True)
        xy = torch.sum(x*y, -1, True)
        num = (1 + 2*self.c*xy + self.c*y2)*x + (1 - self.c*x2)*y
        denom = 1 + 2*self.c*xy + self.c**2 * x2*y2
        return num / torch.clamp(denom, min=1e-15)

    def exp_map(self, p, v):
        v_norm = torch.norm(v, 2, -1, True).clamp(min=1e-10)
        sqrt_c = np.sqrt(self.c)
        if isinstance(p, (int, float)) and p == 0:
            return torch.tanh(sqrt_c * v_norm) * v / (sqrt_c * v_norm)
        
        p_norm_sq = torch.sum(p*p, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        v_at_origin = torch.tanh(sqrt_c * lambda_p * v_norm / 2) * v / (sqrt_c * v_norm)
        return self.mobius_add(p, v_at_origin)

    def log_map(self, p, x):
        if isinstance(p, (int, float)) and p == 0:
            p_tensor = torch.zeros_like(x)
            p_norm_sq = torch.tensor(0.0, device=x.device)
        else:
            p_tensor = p
            p_norm_sq = torch.sum(p*p, -1, True)

        x_trans = self.mobius_add(-p_tensor, x)
        x_norm = torch.norm(x_trans, 2, -1, True).clamp(min=1e-10)
        sqrt_c = np.sqrt(self.c)
        
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        scalar = (2 / (lambda_p * sqrt_c * x_norm)) * torch.atanh((sqrt_c * x_norm).clamp(max=1-1e-7))
        return scalar * x_trans

    def dist(self, x, y):
        sq_dist = torch.norm(x - y, 2, -1)**2
        nx_v = 1 - self.c * torch.norm(x, 2, -1)**2
        ny_v = 1 - self.c * torch.norm(y, 2, -1)**2
        val = 1 + 2 * self.c * sq_dist / (nx_v.clamp(min=1e-15) * ny_v.clamp(min=1e-15))
        return (1 / np.sqrt(self.c)) * torch.acosh(val.clamp(min=1 + 1e-7))

    def project(self, x):
        norm = torch.norm(x, 2, -1, True)
        max_n = 0.99 / np.sqrt(self.c)
        return torch.where(norm >= max_n, x / norm * max_n, x)

class MultiHeadPoincareAttention(nn.Module):
    def __init__(self, d, h, manifold):
        super().__init__()
        self.h, self.d_h, self.manifold = h, d // h, manifold
        self.q_lin, self.k_lin, self.v_lin = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.out_proj = nn.Linear(d, d)

    def forward(self, x):
        b, s, d = x.shape
        q = self.manifold.exp_map(0, self.q_lin(x).view(b, s, self.h, self.d_h)).transpose(1, 2)
        k = self.manifold.exp_map(0, self.k_lin(x).view(b, s, self.h, self.d_h)).transpose(1, 2)
        v = self.v_lin(x).view(b, s, self.h, self.d_h).transpose(1, 2)
        
        attn = torch.softmax(-self.manifold.dist(q.unsqueeze(3), k.unsqueeze(2)) / 1.0, -1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        return self.out_proj(self.manifold.exp_map(0, out))

class PoincareTransformerBlock(nn.Module):
    def __init__(self, d, h, manifold):
        super().__init__()
        self.attn, self.ln1 = MultiHeadPoincareAttention(d, h, manifold), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
        self.ln2, self.manifold = nn.LayerNorm(d), manifold

    def forward(self, x):
        res_attn = self.attn(self.ln1(x))
        x = self.manifold.exp_map(0, self.manifold.log_map(0, x) + res_attn)
        res_ff = self.ff(self.ln2(x))
        x = self.manifold.exp_map(0, self.manifold.log_map(0, x) + res_ff)
        return x

class RSGD(torch.optim.Optimizer):
    def __init__(self, params, lr, manifold):
        super().__init__(params, dict(lr=lr, manifold=manifold))
    def step(self):
        for group in self.param_groups:
            m, lr = group['manifold'], group['lr']
            for p in group['params']:
                if p.grad is None: continue
                p.data = m.project(m.exp_map(p.data, -lr * p.grad.data * ((1 - m.c * torch.sum(p.data**2, -1, True))**2 / 4)))

G = nx.balanced_tree(r=3, h=4)
edge_index = torch.tensor(list(G.edges()), dtype=torch.long)
vocab_size, embed_dim, num_heads = len(G.nodes()), 16, 4
manifold = PoincareManifold()

emb_layer = nn.Embedding(vocab_size, embed_dim)
with torch.no_grad():
    emb_layer.weight.data.uniform_(-0.02, 0.02)

model = PoincareTransformerBlock(embed_dim, num_heads, manifold)
optimizer = RSGD(list(emb_layer.parameters()) + list(model.parameters()), lr=0.5, manifold=manifold)

for epoch in range(1201):
    optimizer.zero_grad()
    x_ball = manifold.exp_map(0, emb_layer(torch.arange(vocab_size)).unsqueeze(0))
    res = model(x_ball)
    
    u, v = edge_index[:, 0], edge_index[:, 1]
    pos_dist = manifold.dist(res[0, u], res[0, v])
    neg_idx = torch.randint(0, vocab_size, (u.size(0),))
    neg_dist = manifold.dist(res[0, u], res[0, neg_idx])
    
    root_penalty = manifold.dist(res[0, 0], torch.zeros_like(res[0, 0])).mean()
    
    loss = pos_dist.mean() - 2.0 * torch.log(neg_dist.mean() + 1e-6) + 1.0 * root_penalty
    loss.backward()
    optimizer.step()
    
    if epoch == 800:
        for g in optimizer.param_groups: g['lr'] *= 0.1
    
    if epoch % 300 == 0: print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

with torch.no_grad():
    final = model(manifold.exp_map(0, emb_layer(torch.arange(vocab_size)).unsqueeze(0))).squeeze(0)
    p_dists, g_dists = [], []
    for _ in range(500):
        u, v = np.random.choice(vocab_size, 2, replace=False)
        g_dists.append(nx.shortest_path_length(G, u, v))
        p_dists.append(manifold.dist(final[u], final[v]).item())
    
    print("\n" + "="*40)
    print(f"RIEMANNIAN RESULTS")
    print("="*40)
    print(f"Correlation: {np.corrcoef(g_dists, p_dists)[0, 1]:.4f}")
    print(f"Max Radius: {torch.norm(final, p=2, dim=-1).max().item():.4f}")
    print("="*40)

Epoch 0 | Loss: 3.0162
Epoch 300 | Loss: -2.0252
Epoch 600 | Loss: -2.4262
Epoch 900 | Loss: -3.1745
Epoch 1200 | Loss: -3.1247

RIEMANNIAN RESULTS
Correlation: 0.6765
Max Radius: 0.9963


# Euclidian Embedded Transformer

In [10]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
import time

G = nx.balanced_tree(r=3, h=4)
edge_index = torch.tensor(list(G.edges()), dtype=torch.long)
vocab_size = len(G.nodes())
embed_dim, num_heads = 16, 4

class EuclideanTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            dim_feedforward=4*embed_dim,
            batch_first=True,
            norm_first=True
        )

    def forward(self, x_idx):
        x = self.embedding(x_idx)
        return self.encoder_layer(x)

model = EuclideanTransformer(vocab_size, embed_dim, num_heads)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(1001):
    optimizer.zero_grad()
    
    all_indices = torch.arange(vocab_size).unsqueeze(0)
    embeddings = model(all_indices).squeeze(0)
    
    u_idx, v_idx = edge_index[:, 0], edge_index[:, 1]
    dist_pos = torch.norm(embeddings[u_idx] - embeddings[v_idx], p=2, dim=-1)
    
    neg_idx = torch.randint(0, vocab_size, (u_idx.size(0),))
    dist_neg = torch.norm(embeddings[u_idx] - embeddings[neg_idx], p=2, dim=-1)
    
    root_penalty = torch.norm(embeddings[0]) 
    
    loss = dist_pos.mean() - 4.0 * torch.log(dist_neg.mean() + 1e-6) + 0.1 * root_penalty
    
    loss.backward()
    optimizer.step()
    
    if epoch % 200 == 0:
        print(f"Epoch {epoch}: Loss {loss.item():.4f}")

def run_final_test(model, graph):
    model.eval()
    with torch.no_grad():
        all_indices = torch.arange(vocab_size).unsqueeze(0)
        res = model(all_indices).squeeze(0)
        
        node_pairs = np.random.choice(vocab_size, (500, 2))
        e_dists, g_dists = [], []
        for u, v in node_pairs:
            if u == v: continue
            g_dists.append(nx.shortest_path_length(graph, source=u, target=v))
            e_dists.append(torch.norm(res[u] - res[v], p=2).item())
        
        corr = np.corrcoef(g_dists, e_dists)[0, 1]
        print("\n" + "="*40)
        print(f"EUCLIDEAN RESULTS")
        print("="*40)
        print(f"Correlation: {corr:.4f}")
        print(f"Max Magnitude: {torch.norm(res, p=2, dim=-1).max().item():.4f}")
        print("="*40)

run_final_test(model, G)

Epoch 0: Loss -0.5116
Epoch 200: Loss -2.8695
Epoch 400: Loss -5.3940
Epoch 600: Loss -5.9397
Epoch 800: Loss -6.9154
Epoch 1000: Loss -7.7305

EUCLIDEAN RESULTS
Correlation: 0.2999
Max Magnitude: 27.7483


## Comparison on Geometric Capacity

In [14]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
import time

class PoincareManifold:
    def __init__(self, c=1.0): self.c = c
    def mobius_add(self, x, y):
        x2, y2, xy = torch.sum(x*x, -1, True), torch.sum(y*y, -1, True), torch.sum(x*y, -1, True)
        num = (1 + 2*self.c*xy + self.c*y2)*x + (1 - self.c*x2)*y
        denom = 1 + 2*self.c*xy + self.c**2 * x2*y2
        return num / torch.clamp(denom, min=1e-15)
    def exp_map(self, p, v):
        v_norm = torch.norm(v, 2, -1, True).clamp(min=1e-10)
        if isinstance(p, (int, float)) and p == 0:
            return torch.tanh(np.sqrt(self.c) * v_norm) * v / (np.sqrt(self.c) * v_norm)
        p_norm_sq = torch.sum(p*p, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        v_at_origin = torch.tanh(np.sqrt(self.c) * lambda_p * v_norm / 2) * v / (np.sqrt(self.c) * v_norm)
        return self.mobius_add(p, v_at_origin)
    def log_map(self, p, x):
        p_tensor = torch.zeros_like(x) if (isinstance(p, (int, float)) and p == 0) else p
        p_norm_sq = torch.sum(p_tensor*p_tensor, -1, True)
        x_trans = self.mobius_add(-p_tensor, x)
        x_norm = torch.norm(x_trans, 2, -1, True).clamp(min=1e-10)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        scalar = (2 / (lambda_p * np.sqrt(self.c) * x_norm)) * torch.atanh((np.sqrt(self.c) * x_norm).clamp(max=1-1e-7))
        return scalar * x_trans
    def dist(self, x, y):
        sq_dist = torch.norm(x - y, 2, -1)**2
        nx_v, ny_v = 1 - self.c * torch.norm(x, 2, -1)**2, 1 - self.c * torch.norm(y, 2, -1)**2
        val = 1 + 2 * self.c * sq_dist / (nx_v.clamp(min=1e-15) * ny_v.clamp(min=1e-15))
        return (1 / np.sqrt(self.c)) * torch.acosh(val.clamp(min=1 + 1e-7))
    def project(self, x):
        norm = torch.norm(x, 2, -1, True)
        return torch.where(norm >= 0.99, x / norm * 0.99, x)

class RSGD(torch.optim.Optimizer):
    def __init__(self, params, lr, manifold): super().__init__(params, dict(lr=lr, manifold=manifold))
    def step(self):
        for group in self.param_groups:
            m, lr = group['manifold'], group['lr']
            for p in group['params']:
                if p.grad is None: continue
                p.data = m.project(m.exp_map(p.data, -lr * p.grad.data * ((1 - m.c * torch.sum(p.data**2, -1, True))**2 / 4)))

class PoincareTransformerBlock(nn.Module):
    def __init__(self, d, h, m):
        super().__init__()
        self.m, self.h, self.d_h = m, h, d // h
        self.q_lin, self.k_lin, self.v_lin = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
    def forward(self, x):
        b, s, d = x.shape
        q = self.m.exp_map(0, self.q_lin(self.ln1(x)).view(b, s, self.h, self.d_h)).transpose(1, 2)
        k = self.m.exp_map(0, self.k_lin(self.ln1(x)).view(b, s, self.h, self.d_h)).transpose(1, 2)
        v = self.v_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        attn = torch.softmax(-self.m.dist(q.unsqueeze(3), k.unsqueeze(2)), -1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        x = self.m.exp_map(0, self.m.log_map(0, x) + self.m.exp_map(0, out))
        x = self.m.exp_map(0, self.m.log_map(0, x) + self.ff(self.ln2(x)))
        return x

class StandardTransformerBlock(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, h, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
    def forward(self, x):
        x = x + self.attn(self.ln1(x), self.ln1(x), self.ln1(x))[0]
        x = x + self.ff(self.ln2(x))
        return x

def run_full_analysis(name, model, emb_layer, manifold, graph, edge_index):
    model.eval(); emb_layer.eval()
    vocab_size = len(graph.nodes())
    indices = torch.arange(vocab_size)
    
    start_time = time.time()
    for _ in range(50):
        with torch.no_grad():
            if name == "RIEMANNIAN":
                res = model(manifold.exp_map(0, emb_layer(indices).unsqueeze(0)))
            else:
                res = model(emb_layer(indices).unsqueeze(0))
    throughput = (vocab_size * 50) / (time.time() - start_time)
    
    res = res.squeeze(0)
    node_pairs = np.random.choice(vocab_size, (500, 2))
    m_dists, g_dists = [], []
    for u, v in node_pairs:
        if u == v: continue
        g_dists.append(nx.shortest_path_length(graph, u, v))
        if name == "RIEMANNIAN":
            m_dists.append(manifold.dist(res[u], res[v]).item())
        else:
            m_dists.append(torch.norm(res[u] - res[v], p=2).item())
    corr = np.corrcoef(g_dists, m_dists)[0, 1]
    
    max_val = torch.norm(res, p=2, dim=-1).max().item()
    
    std_per_dim = res.std(0)
    active_dims = (std_per_dim > 1e-4).sum().item()
    
    print(f"\n[{name} ANALYSIS]")
    print("-" * 30)
    print(f"1. Distance Correlation: {corr:.4f}")
    print(f"2. Max {'Radius' if name == 'RIEMANNIAN' else 'Magnitude'}: {max_val:.4f}")
    print(f"3. Active Dimensions:    {active_dims} / {res.shape[-1]}")
    print(f"4. Throughput:           {throughput:.0f} nodes/sec")

G = nx.balanced_tree(r=3, h=4)
edges = torch.tensor(list(G.edges()), dtype=torch.long)
v_size, d_dim, n_head = len(G.nodes()), 16, 4

m_r = PoincareManifold(); e_r = nn.Embedding(v_size, d_dim)
with torch.no_grad(): e_r.weight.data.uniform_(-0.02, 0.02)
t_r = PoincareTransformerBlock(d_dim, n_head, m_r)
opt_r = RSGD(list(e_r.parameters()) + list(t_r.parameters()), lr=0.5, manifold=m_r)

e_e = nn.Embedding(v_size, d_dim)
t_e = StandardTransformerBlock(d_dim, n_head)
opt_e = torch.optim.Adam(list(e_e.parameters()) + list(t_e.parameters()), lr=0.001)

for epoch in range(1201):
    # Riemannian Step
    opt_r.zero_grad()
    r_out = t_r(m_r.exp_map(0, e_r(torch.arange(v_size)).unsqueeze(0))).squeeze(0)
    loss_r = m_r.dist(r_out[edges[:,0]], r_out[edges[:,1]]).mean() - 2.0*torch.log(m_r.dist(r_out[edges[:,0]], r_out[torch.randint(0,v_size,(edges.size(0),))]).mean()) + 1.0*m_r.dist(r_out[0], torch.zeros_like(r_out[0])).mean()
    loss_r.backward(); opt_r.step()
    
    opt_e.zero_grad()
    e_out = t_e(e_e(torch.arange(v_size)).unsqueeze(0)).squeeze(0)
    loss_e = torch.norm(e_out[edges[:,0]] - e_out[edges[:,1]], p=2, dim=-1).mean() - 2.0*torch.log(torch.norm(e_out[edges[:,0]] - e_out[torch.randint(0,v_size,(edges.size(0),))], p=2, dim=-1).mean()) + 0.1*torch.norm(e_out[0])
    loss_e.backward(); opt_e.step()
    
    if epoch == 800:
        for g in opt_r.param_groups: g['lr'] *= 0.1

run_full_analysis("RIEMANNIAN", t_r, e_r, m_r, G, edges)
run_full_analysis("EUCLIDEAN ", t_e, e_e, None, G, edges)


[RIEMANNIAN ANALYSIS]
------------------------------
1. Distance Correlation: 0.6128
2. Max Radius: 0.9975
3. Active Dimensions:    16 / 16
4. Throughput:           48400 nodes/sec

[EUCLIDEAN  ANALYSIS]
------------------------------
1. Distance Correlation: 0.4282
2. Max Magnitude: 43.6613
3. Active Dimensions:    16 / 16
4. Throughput:           387200 nodes/sec


# Riemannian Embedded Transformer on Taxonomic Data

In [2]:
import sys 
print(sys.executable)

C:\Users\PC\anaconda3\envs\pytorch_env\python.exe


In [2]:
import nltk

nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...


True

In [2]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
from nltk.corpus import wordnet as wn

# --- 1. WORDNET DATA PREPARATION (STABILIZED CONNECTIVITY) ---
def build_wordnet_tree(root_synset_name='mammal.n.01'):
    G = nx.Graph() 
    root = wn.synset(root_synset_name)
    nodes = [root]
    visited = {root}
    
    while nodes:
        current = nodes.pop(0)
        for child in current.hyponyms():
            if child not in visited:
                G.add_edge(current.name(), child.name())
                visited.add(child)
                nodes.append(child)
    
    # NEW: Safety check for disconnected islands
    if not nx.is_connected(G):
        print("Detected disjoint components. Extracting the largest connected component...")
        # Get the largest island (the main tree)
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()
                
    return nx.convert_node_labels_to_integers(G, label_attribute='name')
    
# --- 2. MANIFOLD & TRANSFORMER (STABILIZED) ---
class PoincareManifold:
    def __init__(self, c=1.0): self.c = c
    def mobius_add(self, x, y):
        x2, y2, xy = torch.sum(x*x, -1, True), torch.sum(y*y, -1, True), torch.sum(x*y, -1, True)
        num = (1 + 2*self.c*xy + self.c*y2)*x + (1 - self.c*x2)*y
        denom = 1 + 2*self.c*xy + self.c**2 * x2*y2
        return num / torch.clamp(denom, min=1e-15)
    def exp_map(self, p, v):
        v_norm = torch.norm(v, 2, -1, True).clamp(min=1e-10)
        sqrt_c = np.sqrt(self.c)
        if isinstance(p, (int, float)) and p == 0:
            return torch.tanh(sqrt_c * v_norm) * v / (sqrt_c * v_norm)
        p_norm_sq = torch.sum(p*p, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        v_at_origin = torch.tanh(sqrt_c * lambda_p * v_norm / 2) * v / (sqrt_c * v_norm)
        return self.mobius_add(p, v_at_origin)
    def log_map(self, p, x):
        p_tensor = torch.zeros_like(x) if (isinstance(p, (int, float)) and p == 0) else p
        x_trans = self.mobius_add(-p_tensor, x)
        x_norm = torch.norm(x_trans, 2, -1, True).clamp(min=1e-10)
        p_norm_sq = torch.sum(p_tensor*p_tensor, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        scalar = (2 / (lambda_p * np.sqrt(self.c) * x_norm)) * torch.atanh((np.sqrt(self.c) * x_norm).clamp(max=1-1e-7))
        return scalar * x_trans
    def dist(self, x, y):
        sq_dist = torch.norm(x - y, 2, -1)**2
        nx_v, ny_v = 1 - self.c * torch.norm(x, 2, -1)**2, 1 - self.c * torch.norm(y, 2, -1)**2
        val = 1 + 2 * self.c * sq_dist / (nx_v.clamp(min=1e-15) * ny_v.clamp(min=1e-15))
        return (1 / np.sqrt(self.c)) * torch.acosh(val.clamp(min=1 + 1e-7))
    def project(self, x):
        norm = torch.norm(x, 2, -1, True)
        return torch.where(norm >= 0.99, x / norm * 0.99, x)

class RSGD(torch.optim.Optimizer):
    def __init__(self, params, lr, manifold): super().__init__(params, dict(lr=lr, manifold=manifold))
    def step(self):
        for group in self.param_groups:
            m, lr = group['manifold'], group['lr']
            for p in group['params']:
                if p.grad is None: continue
                p.data = m.project(m.exp_map(p.data, -lr * p.grad.data * ((1 - m.c * torch.sum(p.data**2, -1, True))**2 / 4)))

class PoincareTransformerBlock(nn.Module):
    def __init__(self, d, h, m):
        super().__init__()
        self.m, self.h, self.d_h = m, h, d // h
        self.q_lin, self.k_lin, self.v_lin = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
    def forward(self, x):
        b, s, d = x.shape
        q = self.m.exp_map(0, self.q_lin(self.ln1(x)).view(b, s, self.h, self.d_h)).transpose(1, 2)
        k = self.m.exp_map(0, self.k_lin(self.ln1(x)).view(b, s, self.h, self.d_h)).transpose(1, 2)
        v = self.v_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        attn = torch.softmax(-self.m.dist(q.unsqueeze(3), k.unsqueeze(2)) / 1.0, -1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        x = self.m.exp_map(0, self.m.log_map(0, x) + self.m.exp_map(0, out))
        x = self.m.exp_map(0, self.m.log_map(0, x) + self.ff(self.ln2(x)))
        return x

# --- 3. TRAINING LOOP ---
manifold = PoincareManifold()
emb_layer = nn.Embedding(vocab_size, embed_dim)
with torch.no_grad(): emb_layer.weight.data.uniform_(-0.02, 0.02)
model = PoincareTransformerBlock(embed_dim, num_heads, manifold)
optimizer = RSGD(list(emb_layer.parameters()) + list(model.parameters()), lr=0.5, manifold=manifold)

print("Training WordNet Riemannian Transformer...")
for epoch in range(1501):
    optimizer.zero_grad()
    x_ball = manifold.exp_map(0, emb_layer(torch.arange(vocab_size)).unsqueeze(0))
    res = model(x_ball).squeeze(0)
    
    u, v = edge_index[:, 0], edge_index[:, 1]
    pos_dist = manifold.dist(res[u], res[v])
    neg_dist = manifold.dist(res[u], res[torch.randint(0, vocab_size, (u.size(0),))])
    root_penalty = manifold.dist(res[0], torch.zeros_like(res[0])).mean()
    
    loss = pos_dist.mean() - 2.0 * torch.log(neg_dist.mean() + 1e-6) + 1.0 * root_penalty
    loss.backward()
    optimizer.step()
    
    if epoch == 1000:
        for g in optimizer.param_groups: g['lr'] *= 0.1
    if epoch % 500 == 0: print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

# --- 4. FINAL WORDNET EVALUATION ---
with torch.no_grad():
    final = model(manifold.exp_map(0, emb_layer(torch.arange(vocab_size)).unsqueeze(0))).squeeze(0)
    p_dists, g_dists = [], []
    
    samples_collected = 0
    while samples_collected < 1000:
        u, v = np.random.choice(vocab_size, 2, replace=False)
        try:
            # Shortest path in graph
            gt_dist = nx.shortest_path_length(G, u, v)
            # Hyperbolic distance in manifold
            pred_dist = manifold.dist(final[u], final[v]).item()
            
            g_dists.append(gt_dist)
            p_dists.append(pred_dist)
            samples_collected += 1
        except nx.NetworkXNoPath:
            continue # If we hit a ghost node, just try a different pair
    
    print("\n" + "="*40)
    print(f"WORDNET MAMMALS RESULTS")
    print("="*40)
    print(f"Correlation: {np.corrcoef(g_dists, p_dists)[0, 1]:.4f}")
    print(f"Max Radius:  {torch.norm(final, p=2, dim=-1).max().item():.4f}")
    print("="*40)

# Quick Distance Check: "Dog" vs "Cat" vs "Whale"
# Use mapping['dog.n.01'] to find specific distances if you want!

Training WordNet Riemannian Transformer...
Epoch 0 | Loss: 3.2764
Epoch 500 | Loss: -2.7660
Epoch 1000 | Loss: -2.9189
Epoch 1500 | Loss: -3.5102

WORDNET MAMMALS RESULTS
Correlation: 0.2218
Max Radius:  0.9972


In [1]:
# Gemini claims this is perfect

import torch
import torch.nn as nn
import networkx as nx
import numpy as np
from nltk.corpus import wordnet as wn
import time

# --- 1. DATA PREPARATION (THE FINAL FIX) ---
def build_wordnet_tree(root_synset_name='mammal.n.01'):
    G = nx.Graph() 
    root = wn.synset(root_synset_name)
    nodes = [root]
    visited = {root}
    while nodes:
        current = nodes.pop(0)
        for child in current.hyponyms():
            if child not in visited:
                G.add_edge(current.name(), child.name())
                visited.add(child)
                nodes.append(child)
    
    # SURGERY: Force a single connected component
    if not nx.is_connected(G):
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()
    
    return nx.convert_node_labels_to_integers(G, label_attribute='name')

print("Building WordNet Tree...")
G = build_wordnet_tree()
vocab_size = len(G.nodes())
edge_index = torch.tensor(list(G.edges()), dtype=torch.long)
embed_dim, num_heads = 16, 4

print(f"Dataset Ready: {vocab_size} nodes in a single connected component.")

# --- 2. MANIFOLD & TRANSFORMER ---
class PoincareManifold:
    def __init__(self, c=1.0): self.c = c
    def mobius_add(self, x, y):
        x2, y2, xy = torch.sum(x*x, -1, True), torch.sum(y*y, -1, True), torch.sum(x*y, -1, True)
        num = (1 + 2*self.c*xy + self.c*y2)*x + (1 - self.c*x2)*y
        denom = 1 + 2*self.c*xy + self.c**2 * x2*y2
        return num / torch.clamp(denom, min=1e-15)
    def exp_map(self, p, v):
        v_norm = torch.norm(v, 2, -1, True).clamp(min=1e-10)
        if isinstance(p, (int, float)) and p == 0:
            return torch.tanh(np.sqrt(self.c) * v_norm) * v / (np.sqrt(self.c) * v_norm)
        p_norm_sq = torch.sum(p*p, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        v_at_origin = torch.tanh(np.sqrt(self.c) * lambda_p * v_norm / 2) * v / (np.sqrt(self.c) * v_norm)
        return self.mobius_add(p, v_at_origin)
    def log_map(self, p, x):
        p_tensor = torch.zeros_like(x) if (isinstance(p, (int, float)) and p == 0) else p
        x_trans = self.mobius_add(-p_tensor, x)
        x_norm = torch.norm(x_trans, 2, -1, True).clamp(min=1e-10)
        p_norm_sq = torch.sum(p_tensor*p_tensor, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        scalar = (2 / (lambda_p * np.sqrt(self.c) * x_norm)) * torch.atanh((np.sqrt(self.c) * x_norm).clamp(max=1-1e-7))
        return scalar * x_trans
    def dist(self, x, y):
        sq_dist = torch.norm(x - y, 2, -1)**2
        nx_v, ny_v = 1 - self.c * torch.norm(x, 2, -1)**2, 1 - self.c * torch.norm(y, 2, -1)**2
        val = 1 + 2 * self.c * sq_dist / (nx_v.clamp(min=1e-15) * ny_v.clamp(min=1e-15))
        return (1 / np.sqrt(self.c)) * torch.acosh(val.clamp(min=1 + 1e-7))
    def project(self, x):
        norm = torch.norm(x, 2, -1, True)
        return torch.where(norm >= 0.99, x / norm * 0.99, x)

class RSGD(torch.optim.Optimizer):
    def __init__(self, params, lr, manifold): super().__init__(params, dict(lr=lr, manifold=manifold))
    def step(self):
        for group in self.param_groups:
            m, lr = group['manifold'], group['lr']
            for p in group['params']:
                if p.grad is None: continue
                p.data = m.project(m.exp_map(p.data, -lr * p.grad.data * ((1 - m.c * torch.sum(p.data**2, -1, True))**2 / 4)))

class PoincareTransformerBlock(nn.Module):
    def __init__(self, d, h, m):
        super().__init__()
        self.m, self.h, self.d_h = m, h, d // h
        self.q_lin, self.k_lin, self.v_lin = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
    def forward(self, x):
        b, s, d = x.shape
        q = self.m.exp_map(0, self.q_lin(self.ln1(x)).view(b, s, self.h, self.d_h)).transpose(1, 2)
        k = self.m.exp_map(0, self.k_lin(self.ln1(x)).view(b, s, self.h, self.d_h)).transpose(1, 2)
        v = self.v_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        attn = torch.softmax(-self.m.dist(q.unsqueeze(3), k.unsqueeze(2)) / 1.0, -1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        x = self.m.exp_map(0, self.m.log_map(0, x) + self.m.exp_map(0, out))
        x = self.m.exp_map(0, self.m.log_map(0, x) + self.ff(self.ln2(x)))
        return x

# --- 3. TRAINING LOOP ---
manifold = PoincareManifold()
emb_layer = nn.Embedding(vocab_size, embed_dim)
with torch.no_grad(): emb_layer.weight.data.uniform_(-0.02, 0.02)
model = PoincareTransformerBlock(embed_dim, num_heads, manifold)
optimizer = RSGD(list(emb_layer.parameters()) + list(model.parameters()), lr=0.5, manifold=manifold)

print("Training WordNet Riemannian Transformer...")
for epoch in range(1501):
    optimizer.zero_grad()
    x_ball = manifold.exp_map(0, emb_layer(torch.arange(vocab_size)).unsqueeze(0))
    res = model(x_ball).squeeze(0)
    u, v = edge_index[:, 0], edge_index[:, 1]
    pos_dist = manifold.dist(res[u], res[v])
    neg_dist = manifold.dist(res[u], res[torch.randint(0, vocab_size, (u.size(0),))])
    root_penalty = manifold.dist(res[0], torch.zeros_like(res[0])).mean()
    loss = pos_dist.mean() - 2.0 * torch.log(neg_dist.mean() + 1e-6) + 1.0 * root_penalty
    loss.backward()
    optimizer.step()
    if epoch == 1000:
        for g in optimizer.param_groups: g['lr'] *= 0.1
    if epoch % 500 == 0: print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

# --- 4. FINAL EVALUATION (CRASH-PROOF) ---
with torch.no_grad():
    final = model(manifold.exp_map(0, emb_layer(torch.arange(vocab_size)).unsqueeze(0))).squeeze(0)
    p_dists, g_dists = [], []
    print("Evaluating Correlation...")
    samples = 0
    while samples < 1000:
        u, v = np.random.choice(vocab_size, 2, replace=False)
        try:
            g_dists.append(nx.shortest_path_length(G, u, v))
            p_dists.append(manifold.dist(final[u], final[v]).item())
            samples += 1
        except nx.NetworkXNoPath:
            continue
    
    print("\n" + "="*40)
    print(f"WORDNET MAMMALS FINAL RESULTS")
    print("="*40)
    print(f"Correlation: {np.corrcoef(g_dists, p_dists)[0, 1]:.4f}")
    print(f"Max Radius:  {torch.norm(final, p=2, dim=-1).max().item():.4f}")
    print("="*40)

Building WordNet Tree...
Dataset Ready: 1170 nodes in a single connected component.
Training WordNet Riemannian Transformer...
Epoch 0 | Loss: 2.3565
Epoch 500 | Loss: -2.8242
Epoch 1000 | Loss: -2.7870
Epoch 1500 | Loss: -3.3470
Evaluating Correlation...

WORDNET MAMMALS FINAL RESULTS
Correlation: 0.2035
Max Radius:  0.9974


# Euclidian Embedded Transformer on Taxonomic Data

In [7]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
from nltk.corpus import wordnet as wn
import time

def build_wordnet_tree(root_synset_name='mammal.n.01'):
    G = nx.Graph() 
    root = wn.synset(root_synset_name)
    nodes = [root]
    visited = {root}
    while nodes:
        current = nodes.pop(0)
        for child in current.hyponyms():
            if child not in visited:
                G.add_edge(current.name(), child.name())
                visited.add(child)
                nodes.append(child)
    return nx.convert_node_labels_to_integers(G, label_attribute='name')

G = build_wordnet_tree()
edge_index = torch.tensor(list(G.edges()), dtype=torch.long)
vocab_size = len(G.nodes())
embed_dim, num_heads = 16, 4

class StandardTransformerBlock(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, h, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d, 4*d),
            nn.GELU(),
            nn.Linear(4*d, d)
        )
        self.ln1 = nn.LayerNorm(d)
        self.ln2 = nn.LayerNorm(d)

    def forward(self, x):
        x = x + self.attn(self.ln1(x), self.ln1(x), self.ln1(x))[0]
        x = x + self.ff(self.ln2(x))
        return x

emb_layer = nn.Embedding(vocab_size, embed_dim)
model = StandardTransformerBlock(embed_dim, num_heads)
optimizer = torch.optim.Adam(list(emb_layer.parameters()) + list(model.parameters()), lr=0.001)

print(f"Training Euclidean Transformer on {vocab_size} WordNet nodes...")
start_train = time.time()

for epoch in range(1501):
    optimizer.zero_grad()
    res = model(emb_layer(torch.arange(vocab_size)).unsqueeze(0)).squeeze(0)
    
    u, v = edge_index[:, 0], edge_index[:, 1]
    
    pos_dist = torch.norm(res[u] - res[v], p=2, dim=-1)
    neg_dist = torch.norm(res[u] - res[torch.randint(0, vocab_size, (u.size(0),))], p=2, dim=-1)
    
    loss = pos_dist.mean() - 1.0 * torch.log(neg_dist.mean() + 1e-6) + 0.01 * torch.norm(res, p=2, dim=-1).mean()
    
    loss.backward()
    optimizer.step()
    
    if epoch % 500 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    final = model(emb_layer(torch.arange(vocab_size)).unsqueeze(0)).squeeze(0)
    e_dists, g_dists = [], []
    
    t0 = time.time()
    for _ in range(100):
        _ = model(emb_layer(torch.arange(vocab_size)).unsqueeze(0))
    throughput = (vocab_size * 100) / (time.time() - t0)

    for _ in range(1000):
        u, v = np.random.choice(vocab_size, 2, replace=False)
        g_dists.append(nx.shortest_path_length(G, u, v))
        e_dists.append(torch.norm(final[u] - final[v], p=2).item())
    
    corr = np.corrcoef(g_dists, e_dists)[0, 1]
    max_mag = torch.norm(final, p=2, dim=-1).max().item()

    print("\n" + "="*40)
    print(f"WORDNET EUCLIDEAN RESULTS")
    print("="*40)
    print(f"Correlation:    {corr:.4f}")
    print(f"Max Magnitude:  {max_mag:.4f}")
    print(f"Throughput:     {throughput:.0f} nodes/sec")
    print("="*40)

Training Euclidean Transformer on 1170 WordNet nodes...
Epoch 0 | Loss: 4.0780
Epoch 500 | Loss: -0.2751
Epoch 1000 | Loss: -1.7612
Epoch 1500 | Loss: -2.1281

WORDNET EUCLIDEAN RESULTS
Correlation:    0.0531
Max Magnitude:  38.7971
Throughput:     169197 nodes/sec


# Riemannian Embedded Transformer on Scientific Subbranches Data

In [9]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
from nltk.corpus import wordnet as wn
import time

# --- 1. RIEMANNIAN MANIFOLD & OPTIMIZER (STABILIZED VERSION) ---
class PoincareManifold:
    def __init__(self, c=1.0): self.c = c
    
    def mobius_add(self, x, y):
        x2, y2, xy = torch.sum(x*x, -1, True), torch.sum(y*y, -1, True), torch.sum(x*y, -1, True)
        num = (1 + 2*self.c*xy + self.c*y2)*x + (1 - self.c*x2)*y
        denom = 1 + 2*self.c*xy + self.c**2 * x2*y2
        return num / torch.clamp(denom, min=1e-15)
    
    def exp_map(self, p, v):
        v_norm = torch.norm(v, 2, -1, True).clamp(min=1e-10)
        sqrt_c = np.sqrt(self.c)
        if isinstance(p, (int, float)) and p == 0:
            return torch.tanh(sqrt_c * v_norm) * v / (sqrt_c * v_norm)
        p_norm_sq = torch.sum(p*p, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        v_at_origin = torch.tanh(sqrt_c * lambda_p * v_norm / 2) * v / (sqrt_c * v_norm)
        return self.mobius_add(p, v_at_origin)

    def log_map(self, p, x):
        p_tensor = torch.zeros_like(x) if (isinstance(p, (int, float)) and p == 0) else p
        x_trans = self.mobius_add(-p_tensor, x)
        x_norm = torch.norm(x_trans, 2, -1, True).clamp(min=1e-10)
        p_norm_sq = torch.sum(p_tensor*p_tensor, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        scalar = (2 / (lambda_p * np.sqrt(self.c) * x_norm)) * torch.atanh((np.sqrt(self.c) * x_norm).clamp(max=1-1e-7))
        return scalar * x_trans

    def dist(self, x, y):
        sq_dist = torch.norm(x - y, 2, -1)**2
        nx_v, ny_v = 1 - self.c * torch.norm(x, 2, -1)**2, 1 - self.c * torch.norm(y, 2, -1)**2
        val = 1 + 2 * self.c * sq_dist / (nx_v.clamp(min=1e-15) * ny_v.clamp(min=1e-15))
        return (1 / np.sqrt(self.c)) * torch.acosh(val.clamp(min=1 + 1e-7))

    def project(self, x):
        norm = torch.norm(x, 2, -1, True)
        return torch.where(norm >= 0.99, x / norm * 0.99, x)

class RSGD(torch.optim.Optimizer):
    def __init__(self, params, lr, manifold): super().__init__(params, dict(lr=lr, manifold=manifold))
    def step(self):
        for group in self.param_groups:
            m, lr = group['manifold'], group['lr']
            for p in group['params']:
                if p.grad is None: continue
                rescale = ((1 - m.c * torch.sum(p.data**2, -1, True))**2 / 4)
                p.data = m.project(m.exp_map(p.data, -lr * p.grad.data * rescale))

# --- 2. TRANSFORMER ARCHITECTURE ---
class PoincareTransformerBlock(nn.Module):
    def __init__(self, d, h, m):
        super().__init__()
        self.m, self.h, self.d_h = m, h, d // h
        self.q_lin, self.k_lin, self.v_lin = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
    def forward(self, x):
        b, s, d = x.shape
        q = self.m.exp_map(0, self.q_lin(self.ln1(x)).view(b, s, self.h, self.d_h)).transpose(1, 2)
        k = self.m.exp_map(0, self.k_lin(self.ln1(x)).view(b, s, self.h, self.d_h)).transpose(1, 2)
        v = self.v_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        attn = torch.softmax(-self.m.dist(q.unsqueeze(3), k.unsqueeze(2)) / 1.0, -1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        x = self.m.exp_map(0, self.m.log_map(0, x) + self.m.exp_map(0, out))
        x = self.m.exp_map(0, self.m.log_map(0, x) + self.ff(self.ln2(x)))
        return x

# --- 3. BATCH UTILITIES ---
def build_wordnet_tree(root_synset_name):
    G = nx.Graph() 
    try:
        root = wn.synset(root_synset_name)
    except: return None
    nodes, visited = [root], {root}
    while nodes:
        current = nodes.pop(0)
        for child in current.hyponyms():
            if child not in visited:
                G.add_edge(current.name(), child.name())
                visited.add(child)
                nodes.append(child)
    if len(G.nodes()) < 5: return None
    if not nx.is_connected(G):
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()
    return nx.convert_node_labels_to_integers(G, label_attribute='name')

# --- 4. THE 10-DOMAIN VALIDATION LOOP ---
scientific_roots = [
    ('Physics', 'physics.n.01'), ('Biology', 'biology.n.01'),
    ('Chemistry', 'chemistry.n.01'), ('Mathematics', 'mathematics.n.01'),
    ('Astronomy', 'astronomy.n.01'), ('Geology', 'geology.n.01'),
    ('Mechanics', 'mechanics.n.01'), ('Logic', 'logic.n.01'),
    ('Botany', 'botany.n.01'), ('Zoology', 'zoology.n.01')
]

final_results = {}
embed_dim, num_heads = 16, 4

for name, synset_id in scientific_roots:
    print(f"\n[INIT] Running Domain: {name}")
    G = build_wordnet_tree(synset_id)
    if G is None: continue
    
    vocab_size = len(G.nodes())
    edge_index = torch.tensor(list(G.edges()), dtype=torch.long)
    manifold = PoincareManifold()
    emb_layer = nn.Embedding(vocab_size, embed_dim)
    with torch.no_grad(): emb_layer.weight.data.uniform_(-0.02, 0.02)
    
    model = PoincareTransformerBlock(embed_dim, num_heads, manifold)
    optimizer = RSGD(list(emb_layer.parameters()) + list(model.parameters()), lr=0.5, manifold=manifold)

    for epoch in range(1501):
        optimizer.zero_grad()
        x_ball = manifold.exp_map(0, emb_layer(torch.arange(vocab_size)).unsqueeze(0))
        res = model(x_ball).squeeze(0)
        u, v = edge_index[:, 0], edge_index[:, 1]
        pos_dist = manifold.dist(res[u], res[v])
        neg_dist = manifold.dist(res[u], res[torch.randint(0, vocab_size, (u.size(0),))])
        root_penalty = manifold.dist(res[0], torch.zeros_like(res[0])).mean()
        loss = pos_dist.mean() - 2.0 * torch.log(neg_dist.mean() + 1e-6) + 1.0 * root_penalty
        loss.backward()
        optimizer.step()
        if epoch == 1000:
            for g in optimizer.param_groups: g['lr'] *= 0.1
    
    with torch.no_grad():
        final = model(manifold.exp_map(0, emb_layer(torch.arange(vocab_size)).unsqueeze(0))).squeeze(0)
        p_dists, g_dists, samples = [], [], 0
        while samples < 500:
            u, v = np.random.choice(vocab_size, 2, replace=False)
            try:
                g_dists.append(nx.shortest_path_length(G, u, v))
                p_dists.append(manifold.dist(final[u], final[v]).item())
                samples += 1
            except nx.NetworkXNoPath: continue
        
        corr = np.corrcoef(g_dists, p_dists)[0, 1]
        final_results[name] = corr
        print(f"[{name}] Correlation: {corr:.4f} | Size: {vocab_size} nodes")

print("\n" + "="*40 + "\nSUMMARY TABLE FOR PREPRINT\n" + "="*40)
for k, v in final_results.items(): print(f"{k:15} | {v:.4f}")


[INIT] Running Domain: Physics
[Physics] Correlation: 0.6308 | Size: 50 nodes

[INIT] Running Domain: Biology
[Biology] Correlation: 0.5660 | Size: 74 nodes

[INIT] Running Domain: Chemistry
[Chemistry] Correlation: 0.7478 | Size: 16 nodes

[INIT] Running Domain: Mathematics
[Mathematics] Correlation: 0.6196 | Size: 50 nodes

[INIT] Running Domain: Astronomy
[Astronomy] Correlation: 0.6797 | Size: 9 nodes

[INIT] Running Domain: Geology
[Geology] Correlation: 0.6440 | Size: 19 nodes

[INIT] Running Domain: Mechanics
[Mechanics] Correlation: 0.5727 | Size: 12 nodes

[INIT] Running Domain: Logic

[INIT] Running Domain: Botany
[Botany] Correlation: 0.5671 | Size: 28 nodes

[INIT] Running Domain: Zoology

SUMMARY TABLE FOR PREPRINT
Physics         | 0.6308
Biology         | 0.5660
Chemistry       | 0.7478
Mathematics     | 0.6196
Astronomy       | 0.6797
Geology         | 0.6440
Mechanics       | 0.5727
Botany          | 0.5671


# Euclidian Embedded Transformer on Scientific Subbranches Data

In [6]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
from nltk.corpus import wordnet as wn

# --- 1. EUCLIDEAN UTILITIES ---
class EuclideanSpace:
    """Standard Flat Geometry for Comparison"""
    def dist(self, x, y):
        # Standard L2 (Euclidean) Distance
        return torch.norm(x - y, p=2, dim=-1)

# --- 2. EUCLIDEAN TRANSFORMER ---
class EuclideanTransformerBlock(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.d_h = h, d // h
        self.q_lin = nn.Linear(d, d)
        self.k_lin = nn.Linear(d, d)
        self.v_lin = nn.Linear(d, d)
        self.ln1 = nn.LayerNorm(d)
        self.ln2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(
            nn.Linear(d, 4*d),
            nn.GELU(),
            nn.Linear(4*d, d)
        )
        
    def forward(self, x):
        b, s, d = x.shape
        # Standard Linear Projections
        q = self.q_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        k = self.k_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        v = self.v_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        
        # Standard Dot-Product Attention
        # (Transformed to distance-based for fair comparison with Riemannian version)
        dist_sq = torch.cdist(q, k, p=2)**2
        attn = torch.softmax(-dist_sq / np.sqrt(self.d_h), dim=-1)
        
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        x = x + out # Standard Residual
        x = x + self.ff(self.ln2(x))
        return x

# --- 3. BATCH UTILITIES ---
def build_wordnet_tree(root_synset_name):
    G = nx.Graph() 
    try:
        root = wn.synset(root_synset_name)
    except: return None
    nodes, visited = [root], {root}
    while nodes:
        current = nodes.pop(0)
        for child in current.hyponyms():
            if child not in visited:
                G.add_edge(current.name(), child.name())
                visited.add(child)
                nodes.append(child)
    if len(G.nodes()) < 15: return None
    if not nx.is_connected(G):
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()
    return nx.convert_node_labels_to_integers(G, label_attribute='name')

# --- 4. BATCH EVALUATION LOOP ---
scientific_roots = [
    ('Physics', 'physics.n.01'), ('Biology', 'biology.n.01'),
    ('Chemistry', 'chemistry.n.01'), ('Mathematics', 'mathematics.n.01'),
    ('Astronomy', 'astronomy.n.01'), ('Geology', 'geology.n.01'),
    ('Mechanics', 'mechanics.n.01'), ('Logic', 'logic.n.01'),
    ('Botany', 'botany.n.01'), ('Zoology', 'zoology.n.01')
]

euclidean_results = {}
embed_dim, num_heads = 16, 4

for name, synset_id in scientific_roots:
    print(f"\n[INIT] Running Euclidean Baseline: {name}")
    G = build_wordnet_tree(synset_id)
    if G is None: continue
    
    vocab_size = len(G.nodes())
    edge_index = torch.tensor(list(G.edges()), dtype=torch.long)
    
    # Standard Embedding and Transformer
    emb_layer = nn.Embedding(vocab_size, embed_dim)
    model = EuclideanTransformerBlock(embed_dim, num_heads)
    space = EuclideanSpace()
    
    # Standard Adam Optimizer
    optimizer = torch.optim.Adam(list(emb_layer.parameters()) + list(model.parameters()), lr=0.01)

    for epoch in range(1501):
        optimizer.zero_grad()
        # No exp_map needed, just raw embeddings
        res = model(emb_layer(torch.arange(vocab_size)).unsqueeze(0)).squeeze(0)
        
        u, v = edge_index[:, 0], edge_index[:, 1]
        pos_dist = space.dist(res[u], res[v])
        # Negative sampling to separate unrelated nodes
        neg_dist = space.dist(res[u], res[torch.randint(0, vocab_size, (u.size(0),))])
        
        loss = pos_dist.mean() - torch.log(neg_dist.mean() + 1e-6)
        loss.backward()
        optimizer.step()
        
    with torch.no_grad():
        final = model(emb_layer(torch.arange(vocab_size)).unsqueeze(0)).squeeze(0)
        p_dists, g_dists, samples = [], [], 0
        while samples < 500:
            u, v = np.random.choice(vocab_size, 2, replace=False)
            try:
                g_dists.append(nx.shortest_path_length(G, u, v))
                p_dists.append(space.dist(final[u], final[v]).item())
                samples += 1
            except nx.NetworkXNoPath: continue
        
        corr = np.corrcoef(g_dists, p_dists)[0, 1]
        euclidean_results[name] = corr
        print(f"[{name}] Euclidean Correlation: {corr:.4f}")

print("\n" + "="*40 + "\nEUCLIDEAN BASELINE SUMMARY\n" + "="*40)
for k, v in euclidean_results.items(): print(f"{k:15} | {v:.4f}")


[INIT] Running Euclidean Baseline: Physics
[Physics] Euclidean Correlation: 0.3955

[INIT] Running Euclidean Baseline: Biology
[Biology] Euclidean Correlation: 0.4879

[INIT] Running Euclidean Baseline: Chemistry
[Chemistry] Euclidean Correlation: 0.7223

[INIT] Running Euclidean Baseline: Mathematics
[Mathematics] Euclidean Correlation: 0.7152

[INIT] Running Euclidean Baseline: Astronomy

[INIT] Running Euclidean Baseline: Geology
[Geology] Euclidean Correlation: 0.5903

[INIT] Running Euclidean Baseline: Mechanics

[INIT] Running Euclidean Baseline: Logic

[INIT] Running Euclidean Baseline: Botany
[Botany] Euclidean Correlation: 0.5114

[INIT] Running Euclidean Baseline: Zoology

EUCLIDEAN BASELINE SUMMARY
Physics         | 0.3955
Biology         | 0.4879
Chemistry       | 0.7223
Mathematics     | 0.7152
Geology         | 0.5903
Botany          | 0.5114


# Riemannian Embedded Transformer on Biological Taxonomic Data

In [10]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
from nltk.corpus import wordnet as wn

# --- 1. RIEMANNIAN MANIFOLD & OPTIMIZER (STABILIZED) ---
class PoincareManifold:
    def __init__(self, c=1.0): self.c = c
    def mobius_add(self, x, y):
        x2, y2, xy = torch.sum(x*x, -1, True), torch.sum(y*y, -1, True), torch.sum(x*y, -1, True)
        num = (1 + 2*self.c*xy + self.c*y2)*x + (1 - self.c*x2)*y
        denom = 1 + 2*self.c*xy + self.c**2 * x2*y2
        return num / torch.clamp(denom, min=1e-15)
    
    def exp_map(self, p, v):
        v_norm = torch.norm(v, 2, -1, True).clamp(min=1e-10)
        sqrt_c = np.sqrt(self.c)
        if isinstance(p, (int, float)) and p == 0:
            return torch.tanh(sqrt_c * v_norm) * v / (sqrt_c * v_norm)
        p_norm_sq = torch.sum(p*p, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        v_at_origin = torch.tanh(sqrt_c * lambda_p * v_norm / 2) * v / (sqrt_c * v_norm)
        return self.mobius_add(p, v_at_origin)

    def log_map(self, p, x):
        p_tensor = torch.zeros_like(x) if (isinstance(p, (int, float)) and p == 0) else p
        x_trans = self.mobius_add(-p_tensor, x)
        x_norm = torch.norm(x_trans, 2, -1, True).clamp(min=1e-10)
        p_norm_sq = torch.sum(p_tensor*p_tensor, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        scalar = (2 / (lambda_p * np.sqrt(self.c) * x_norm)) * torch.atanh((np.sqrt(self.c) * x_norm).clamp(max=1-1e-7))
        return scalar * x_trans

    def dist(self, x, y):
        sq_dist = torch.norm(x - y, 2, -1)**2
        nx_v, ny_v = 1 - self.c * torch.norm(x, 2, -1)**2, 1 - self.c * torch.norm(y, 2, -1)**2
        val = 1 + 2 * self.c * sq_dist / (nx_v.clamp(min=1e-15) * ny_v.clamp(min=1e-15))
        return (1 / np.sqrt(self.c)) * torch.acosh(val.clamp(min=1 + 1e-7))

    def project(self, x):
        norm = torch.norm(x, 2, -1, True)
        return torch.where(norm >= 0.99, x / norm * 0.99, x)

class RSGD(torch.optim.Optimizer):
    def __init__(self, params, lr, manifold): super().__init__(params, dict(lr=lr, manifold=manifold))
    def step(self):
        for group in self.param_groups:
            m, lr = group['manifold'], group['lr']
            for p in group['params']:
                if p.grad is None: continue
                rescale = ((1 - m.c * torch.sum(p.data**2, -1, True))**2 / 4)
                p.data = m.project(m.exp_map(p.data, -lr * p.grad.data * rescale))

# --- 2. TRANSFORMER BLOCK ---
class PoincareTransformerBlock(nn.Module):
    def __init__(self, d, h, m):
        super().__init__()
        self.m, self.h, self.d_h = m, h, d // h
        self.q_lin, self.k_lin, self.v_lin = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
    def forward(self, x):
        b, s, d = x.shape
        q = self.m.exp_map(0, self.q_lin(self.ln1(x)).view(b, s, self.h, self.d_h)).transpose(1, 2)
        k = self.m.exp_map(0, self.k_lin(self.ln1(x)).view(b, s, self.h, self.d_h)).transpose(1, 2)
        v = self.v_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        attn = torch.softmax(-self.m.dist(q.unsqueeze(3), k.unsqueeze(2)) / 1.0, -1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        x = self.m.exp_map(0, self.m.log_map(0, x) + self.m.exp_map(0, out))
        x = self.m.exp_map(0, self.m.log_map(0, x) + self.ff(self.ln2(x)))
        return x

# --- 3. BIOLOGICAL DATA PREPARATION ---
def build_bio_taxonomy(root_name='living_thing.n.01', max_nodes=500):
    G = nx.Graph()
    root = wn.synset(root_name)
    nodes, visited = [root], {root}
    
    while nodes and len(G.nodes()) < max_nodes:
        current = nodes.pop(0)
        for child in current.hyponyms():
            if child not in visited:
                G.add_edge(current.name(), child.name())
                visited.add(child)
                nodes.append(child)
                
    if not nx.is_connected(G):
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()
    
    return nx.convert_node_labels_to_integers(G, label_attribute='name'), G

print("Crawling WordNet for Biological Taxonomy (Living Things)...")
mapping_G, raw_G = build_bio_taxonomy()
vocab_size = len(mapping_G.nodes())
edge_index = torch.tensor(list(mapping_G.edges()), dtype=torch.long)
print(f"Dataset Built: {vocab_size} biological entities.")

# --- 4. TRAINING LOOP ---
embed_dim, num_heads = 16, 4
manifold = PoincareManifold()
emb_layer = nn.Embedding(vocab_size, embed_dim)
with torch.no_grad(): emb_layer.weight.data.uniform_(-0.02, 0.02)
model = PoincareTransformerBlock(embed_dim, num_heads, manifold)
optimizer = RSGD(list(emb_layer.parameters()) + list(model.parameters()), lr=0.5, manifold=manifold)

print("Training Riemannian Biological Transformer...")
for epoch in range(1501):
    optimizer.zero_grad()
    x_ball = manifold.exp_map(0, emb_layer(torch.arange(vocab_size)).unsqueeze(0))
    res = model(x_ball).squeeze(0)
    
    u, v = edge_index[:, 0], edge_index[:, 1]
    pos_dist = manifold.dist(res[u], res[v])
    neg_dist = manifold.dist(res[u], res[torch.randint(0, vocab_size, (u.size(0),))])
    root_penalty = manifold.dist(res[0], torch.zeros_like(res[0])).mean()
    
    # Loss optimized for large taxonomies
    loss = pos_dist.mean() - 2.0 * torch.log(neg_dist.mean() + 1e-6) + 1.0 * root_penalty
    loss.backward()
    optimizer.step()
    
    if epoch == 1000:
        for g in optimizer.param_groups: g['lr'] *= 0.1
    if epoch % 500 == 0: print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

# --- 5. EVALUATION ---
with torch.no_grad():
    final = model(manifold.exp_map(0, emb_layer(torch.arange(vocab_size)).unsqueeze(0))).squeeze(0)
    p_dists, g_dists, samples = [], [], 0
    while samples < 1000:
        u, v = np.random.choice(vocab_size, 2, replace=False)
        try:
            g_dists.append(nx.shortest_path_length(mapping_G, u, v))
            p_dists.append(manifold.dist(final[u], final[v]).item())
            samples += 1
        except nx.NetworkXNoPath: continue
        
    print("\n" + "="*40)
    print(f"BIOLOGICAL TAXONOMY RESULTS")
    print("="*40)
    print(f"Correlation: {np.corrcoef(g_dists, p_dists)[0, 1]:.4f}")
    print(f"Max Radius:  {torch.norm(final, p=2, dim=-1).max().item():.4f}")
    print("="*40)

Crawling WordNet for Biological Taxonomy (Living Things)...
Dataset Built: 646 biological entities.
Training Riemannian Biological Transformer...
Epoch 0 | Loss: 2.0058
Epoch 500 | Loss: -2.1355
Epoch 1000 | Loss: -2.6913
Epoch 1500 | Loss: -3.3578

BIOLOGICAL TAXONOMY RESULTS
Correlation: 0.8822
Max Radius:  0.9985


# Euclidian Embedded Transformer on Biological Taxonomic Data

In [12]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
from nltk.corpus import wordnet as wn

# --- 1. EUCLIDEAN UTILITIES ---
class EuclideanSpace:
    """Standard Flat Geometry for Baseline Comparison"""
    def dist(self, x, y):
        # Standard L2 (Euclidean) Distance
        return torch.norm(x - y, p=2, dim=-1)

# --- 2. EUCLIDEAN TRANSFORMER BLOCK ---
class EuclideanTransformerBlock(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.d_h = h, d // h
        self.q_lin = nn.Linear(d, d)
        self.k_lin = nn.Linear(d, d)
        self.v_lin = nn.Linear(d, d)
        self.ln1 = nn.LayerNorm(d)
        self.ln2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(
            nn.Linear(d, 4*d),
            nn.GELU(),
            nn.Linear(4*d, d)
        )
        
    def forward(self, x):
        b, s, d = x.shape
        # Standard Linear Projections
        q = self.q_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        k = self.k_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        v = self.v_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        
        # Standard Attention (scaled dot-product based on Euclidean similarity)
        # Using negative distance to align with the Riemannian logic
        dist_sq = torch.cdist(q, k, p=2)**2
        attn = torch.softmax(-dist_sq / np.sqrt(self.d_h), dim=-1)
        
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        x = x + out # Standard Residual Connection
        x = x + self.ff(self.ln2(x))
        return x

# --- 3. BIOLOGICAL DATA PREPARATION ---
def build_bio_taxonomy(root_name='living_thing.n.01', max_nodes=500):
    G = nx.Graph()
    try:
        root = wn.synset(root_name)
    except:
        print(f"Error: Synset {root_name} not found. Ensure NLTK WordNet is downloaded.")
        return None, None
        
    nodes, visited = [root], {root}
    while nodes and len(G.nodes()) < max_nodes:
        current = nodes.pop(0)
        for child in current.hyponyms():
            if child not in visited:
                G.add_edge(current.name(), child.name())
                visited.add(child)
                nodes.append(child)
                
    if not nx.is_connected(G):
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()
    
    return nx.convert_node_labels_to_integers(G, label_attribute='name'), G

print("Crawling WordNet for Biological Taxonomy Baseline (Euclidean)...")
mapping_G, raw_G = build_bio_taxonomy()
if mapping_G:
    vocab_size = len(mapping_G.nodes())
    edge_index = torch.tensor(list(mapping_G.edges()), dtype=torch.long)
    print(f"Dataset Built: {vocab_size} biological entities.")

    # --- 4. TRAINING LOOP ---
    embed_dim, num_heads = 16, 4
    space = EuclideanSpace()
    emb_layer = nn.Embedding(vocab_size, embed_dim)
    model = EuclideanTransformerBlock(embed_dim, num_heads)
    
    # Standard Adam Optimizer for Euclidean space
    optimizer = torch.optim.Adam(list(emb_layer.parameters()) + list(model.parameters()), lr=0.01)

    print("Training Euclidean Biological Transformer...")
    for epoch in range(1501):
        optimizer.zero_grad()
        # Raw embeddings in flat space
        res = model(emb_layer(torch.arange(vocab_size)).unsqueeze(0)).squeeze(0)
        
        u, v = edge_index[:, 0], edge_index[:, 1]
        pos_dist = space.dist(res[u], res[v])
        neg_dist = space.dist(res[u], res[torch.randint(0, vocab_size, (u.size(0),))])
        
        # Loss formula mirrors the Riemannian run for a fair benchmark
        loss = pos_dist.mean() - 2.0 * torch.log(neg_dist.mean() + 1e-6)
        loss.backward()
        optimizer.step()
        
        if epoch % 500 == 0: 
            print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

    # --- 5. EVALUATION ---
    with torch.no_grad():
        final = model(emb_layer(torch.arange(vocab_size)).unsqueeze(0)).squeeze(0)
        e_dists, g_dists, samples = [], [], 0
        while samples < 1000:
            u, v = np.random.choice(vocab_size, 2, replace=False)
            try:
                g_dists.append(nx.shortest_path_length(mapping_G, u, v))
                e_dists.append(space.dist(final[u], final[v]).item())
                samples += 1
            except nx.NetworkXNoPath: 
                continue
            
        print("\n" + "="*40)
        print(f"BIOLOGICAL TAXONOMY BASELINE (EUCLIDEAN)")
        print("="*40)
        print(f"Correlation: {np.corrcoef(g_dists, e_dists)[0, 1]:.4f}")
        print(f"Max Embedding Norm: {torch.norm(final, p=2, dim=-1).max().item():.4f}")
        print("="*40)

Crawling WordNet for Biological Taxonomy Baseline (Euclidean)...
Dataset Built: 646 biological entities.
Training Euclidean Biological Transformer...
Epoch 0 | Loss: 2.5108
Epoch 500 | Loss: -10.6103
Epoch 1000 | Loss: -10.8393
Epoch 1500 | Loss: -10.8831

BIOLOGICAL TAXONOMY BASELINE (EUCLIDEAN)
Correlation: 0.6908
Max Embedding Norm: 1122.9119


# Riemannian Embedded Transformer on Family / Kinship Data

In [13]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
from nltk.corpus import wordnet as wn

# --- 1. MANIFOLDS ---
class PoincareManifold:
    def __init__(self, c=1.0): self.c = c
    def mobius_add(self, x, y):
        x2, y2, xy = torch.sum(x*x, -1, True), torch.sum(y*y, -1, True), torch.sum(x*y, -1, True)
        num = (1 + 2*self.c*xy + self.c*y2)*x + (1 - self.c*x2)*y
        denom = 1 + 2*self.c*xy + self.c**2 * x2*y2
        return num / torch.clamp(denom, min=1e-15)
    def exp_map(self, p, v):
        v_norm = torch.norm(v, 2, -1, True).clamp(min=1e-10)
        sqrt_c = np.sqrt(self.c)
        if isinstance(p, (int, float)) and p == 0:
            return torch.tanh(sqrt_c * v_norm) * v / (sqrt_c * v_norm)
        p_norm_sq = torch.sum(p*p, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        v_at_origin = torch.tanh(sqrt_c * lambda_p * v_norm / 2) * v / (sqrt_c * v_norm)
        return self.mobius_add(p, v_at_origin)
    def log_map(self, p, x):
        p_tensor = torch.zeros_like(x) if (isinstance(p, (int, float)) and p == 0) else p
        x_trans = self.mobius_add(-p_tensor, x)
        x_norm = torch.norm(x_trans, 2, -1, True).clamp(min=1e-10)
        p_norm_sq = torch.sum(p_tensor*p_tensor, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        scalar = (2 / (lambda_p * np.sqrt(self.c) * x_norm)) * torch.atanh((np.sqrt(self.c) * x_norm).clamp(max=1-1e-7))
        return scalar * x_trans
    def dist(self, x, y):
        sq_dist = torch.norm(x - y, 2, -1)**2
        nx_v, ny_v = 1 - self.c * torch.norm(x, 2, -1)**2, 1 - self.c * torch.norm(y, 2, -1)**2
        val = 1 + 2 * self.c * sq_dist / (nx_v.clamp(min=1e-15) * ny_v.clamp(min=1e-15))
        return (1 / np.sqrt(self.c)) * torch.acosh(val.clamp(min=1 + 1e-7))
    def project(self, x):
        norm = torch.norm(x, 2, -1, True)
        return torch.where(norm >= 0.99, x / norm * 0.99, x)

class RSGD(torch.optim.Optimizer):
    def __init__(self, params, lr, manifold): super().__init__(params, dict(lr=lr, manifold=manifold))
    def step(self):
        for group in self.param_groups:
            m, lr = group['manifold'], group['lr']
            for p in group['params']:
                if p.grad is None: continue
                rescale = ((1 - m.c * torch.sum(p.data**2, -1, True))**2 / 4)
                p.data = m.project(m.exp_map(p.data, -lr * p.grad.data * rescale))

# --- 2. TRANSFORMER BLOCKS ---
class PoincareTransformerBlock(nn.Module):
    def __init__(self, d, h, m):
        super().__init__()
        self.m, self.h, self.d_h = m, h, d // h
        self.q_lin, self.k_lin, self.v_lin = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
    def forward(self, x):
        b, s, d = x.shape
        q = self.m.exp_map(0, self.q_lin(self.ln1(x)).view(b, s, self.h, self.d_h)).transpose(1, 2)
        k = self.m.exp_map(0, self.k_lin(self.ln1(x)).view(b, s, self.h, self.d_h)).transpose(1, 2)
        v = self.v_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        attn = torch.softmax(-self.m.dist(q.unsqueeze(3), k.unsqueeze(2)) / 1.0, -1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        x = self.m.exp_map(0, self.m.log_map(0, x) + self.m.exp_map(0, out))
        x = self.m.exp_map(0, self.m.log_map(0, x) + self.ff(self.ln2(x)))
        return x

class EuclideanTransformerBlock(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.d_h = h, d // h
        self.q_lin, self.k_lin, self.v_lin = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
    def forward(self, x):
        b, s, d = x.shape
        q = self.q_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        k = self.k_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        v = self.v_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        dist_sq = torch.cdist(q, k, p=2)**2
        attn = torch.softmax(-dist_sq / np.sqrt(self.d_h), dim=-1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        x = x + out
        x = x + self.ff(self.ln2(x))
        return x

# --- 3. KINSHIP DATA ---
def build_kinship_ontology(root_name='kinship.n.01', max_nodes=200):
    G = nx.Graph()
    root = wn.synset(root_name)
    nodes, visited = [root], {root}
    while nodes and len(G.nodes()) < max_nodes:
        current = nodes.pop(0)
        for child in current.hyponyms():
            if child not in visited:
                G.add_edge(current.name(), child.name())
                visited.add(child)
                nodes.append(child)
    if not nx.is_connected(G):
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()
    return nx.convert_node_labels_to_integers(G, label_attribute='name'), G

# --- 4. EXECUTION LOOP ---
embed_dim, num_heads = 16, 4
mapping_G, raw_G = build_kinship_ontology()
vocab_size = len(mapping_G.nodes())
edge_index = torch.tensor(list(mapping_G.edges()), dtype=torch.long)

def run_experiment(mode='riemannian'):
    print(f"\n--- Starting {mode.upper()} Kinship Experiment ---")
    emb = nn.Embedding(vocab_size, embed_dim)
    if mode == 'riemannian':
        m = PoincareManifold()
        model = PoincareTransformerBlock(embed_dim, num_heads, m)
        opt = RSGD(list(emb.parameters()) + list(model.parameters()), lr=0.5, manifold=m)
    else:
        model = EuclideanTransformerBlock(embed_dim, num_heads)
        opt = torch.optim.Adam(list(emb.parameters()) + list(model.parameters()), lr=0.01)

    for epoch in range(1501):
        opt.zero_grad()
        if mode == 'riemannian':
            x = m.exp_map(0, emb(torch.arange(vocab_size)).unsqueeze(0))
            res = model(x).squeeze(0)
            u, v = edge_index[:, 0], edge_index[:, 1]
            pos, neg = m.dist(res[u], res[v]), m.dist(res[u], res[torch.randint(0, vocab_size, (u.size(0),))])
            loss = pos.mean() - 1.5 * torch.log(neg.mean() + 1e-6)
        else:
            res = model(emb(torch.arange(vocab_size)).unsqueeze(0)).squeeze(0)
            u, v = edge_index[:, 0], edge_index[:, 1]
            pos, neg = torch.norm(res[u]-res[v], p=2, dim=-1), torch.norm(res[u]-res[torch.randint(0, vocab_size, (u.size(0),))], p=2, dim=-1)
            loss = pos.mean() - 1.5 * torch.log(neg.mean() + 1e-6)
        
        loss.backward()
        opt.step()
    
    with torch.no_grad():
        if mode == 'riemannian': final = model(m.exp_map(0, emb(torch.arange(vocab_size)).unsqueeze(0))).squeeze(0)
        else: final = model(emb(torch.arange(vocab_size)).unsqueeze(0)).squeeze(0)
        d_p, d_g = [], []
        for _ in range(500):
            u, v = np.random.choice(vocab_size, 2, replace=False)
            d_g.append(nx.shortest_path_length(mapping_G, u, v))
            if mode == 'riemannian': d_p.append(m.dist(final[u], final[v]).item())
            else: d_p.append(torch.norm(final[u]-final[v], p=2).item())
        return np.corrcoef(d_g, d_p)[0, 1]

r_corr = run_experiment('riemannian')
e_corr = run_experiment('euclidean')

print("\n" + "="*35)
print(f"KINSHIP COMPARISON (N={vocab_size})")
print("="*35)
print(f"Riemannian Correlation: {r_corr:.4f}")
print(f"Euclidean Correlation:  {e_corr:.4f}")
print("="*35)


--- Starting RIEMANNIAN Kinship Experiment ---

--- Starting EUCLIDEAN Kinship Experiment ---

KINSHIP COMPARISON (N=4)
Riemannian Correlation: 0.8348
Euclidean Correlation:  0.7412


# Euclidian Embedded Transformer on Family / Kinship Data

In [14]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
from nltk.corpus import wordnet as wn

# --- 1. EUCLIDEAN UTILITIES ---
class EuclideanSpace:
    def dist(self, x, y):
        return torch.norm(x - y, p=2, dim=-1)

# --- 2. EUCLIDEAN TRANSFORMER BLOCK ---
class EuclideanTransformerBlock(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.d_h = h, d // h
        self.q_lin = nn.Linear(d, d)
        self.k_lin = nn.Linear(d, d)
        self.v_lin = nn.Linear(d, d)
        self.ln1 = nn.LayerNorm(d)
        self.ln2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(
            nn.Linear(d, 4*d),
            nn.GELU(),
            nn.Linear(4*d, d)
        )
        
    def forward(self, x):
        b, s, d = x.shape
        q = self.q_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        k = self.k_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        v = self.v_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        
        # Standard scaled dot-product attention
        dist_sq = torch.cdist(q, k, p=2)**2
        attn = torch.softmax(-dist_sq / np.sqrt(self.d_h), dim=-1)
        
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        x = x + out 
        x = x + self.ff(self.ln2(x))
        return x

# --- 3. KINSHIP DATA PREPARATION ---
def build_kinship_ontology(root_name='kinship.n.01', max_nodes=200):
    G = nx.Graph()
    try:
        root = wn.synset(root_name)
    except:
        return None, None
        
    nodes, visited = [root], {root}
    while nodes and len(G.nodes()) < max_nodes:
        current = nodes.pop(0)
        for child in current.hyponyms():
            if child not in visited:
                G.add_edge(current.name(), child.name())
                visited.add(child)
                nodes.append(child)
                
    if not nx.is_connected(G):
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()
    
    return nx.convert_node_labels_to_integers(G, label_attribute='name'), G

print("Building Kinship Baseline (Euclidean)...")
mapping_G, raw_G = build_kinship_ontology()
if mapping_G:
    vocab_size = len(mapping_G.nodes())
    edge_index = torch.tensor(list(mapping_G.edges()), dtype=torch.long)
    print(f"Dataset Built: {vocab_size} kinship terms.")

    # --- 4. TRAINING LOOP ---
    embed_dim, num_heads = 16, 4
    space = EuclideanSpace()
    emb_layer = nn.Embedding(vocab_size, embed_dim)
    model = EuclideanTransformerBlock(embed_dim, num_heads)
    
    optimizer = torch.optim.Adam(list(emb_layer.parameters()) + list(model.parameters()), lr=0.01)

    print("Training Euclidean Kinship Transformer...")
    for epoch in range(1501):
        optimizer.zero_grad()
        res = model(emb_layer(torch.arange(vocab_size)).unsqueeze(0)).squeeze(0)
        
        u, v = edge_index[:, 0], edge_index[:, 1]
        pos_dist = space.dist(res[u], res[v])
        neg_dist = space.dist(res[u], res[torch.randint(0, vocab_size, (u.size(0),))])
        
        loss = pos_dist.mean() - 1.5 * torch.log(neg_dist.mean() + 1e-6)
        loss.backward()
        optimizer.step()
        
        if epoch % 500 == 0: 
            print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

    # --- 5. EVALUATION ---
    with torch.no_grad():
        final = model(emb_layer(torch.arange(vocab_size)).unsqueeze(0)).squeeze(0)
        e_dists, g_dists, samples = [], [], 0
        while samples < 500:
            u, v = np.random.choice(vocab_size, 2, replace=False)
            try:
                g_dists.append(nx.shortest_path_length(mapping_G, u, v))
                e_dists.append(space.dist(final[u], final[v]).item())
                samples += 1
            except nx.NetworkXNoPath: 
                continue
            
        print("\n" + "="*40)
        print(f"KINSHIP BASELINE (EUCLIDEAN)")
        print("="*40)
        print(f"Correlation: {np.corrcoef(g_dists, e_dists)[0, 1]:.4f}")
        print(f"Max Embedding Norm: {torch.norm(final, p=2, dim=-1).max().item():.4f}")
        print("="*40)

Building Kinship Baseline (Euclidean)...
Dataset Built: 4 kinship terms.
Training Euclidean Kinship Transformer...
Epoch 0 | Loss: 2.8408
Epoch 500 | Loss: 0.2156
Epoch 1000 | Loss: 0.8198
Epoch 1500 | Loss: 0.8287

KINSHIP BASELINE (EUCLIDEAN)
Correlation: 0.7270
Max Embedding Norm: 27.5664


# Riemannian Embedded Transformer on Supply Chain Data.

In [17]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np

# --- 1. POINCARÉ MANIFOLD & OPTIMIZER ---
class PoincareManifold:
    def __init__(self, c=1.0): self.c = c
    def mobius_add(self, x, y):
        x2, y2, xy = torch.sum(x*x, -1, True), torch.sum(y*y, -1, True), torch.sum(x*y, -1, True)
        num = (1 + 2*self.c*xy + self.c*y2)*x + (1 - self.c*x2)*y
        denom = 1 + 2*self.c*xy + self.c**2 * x2*y2
        return num / torch.clamp(denom, min=1e-15)
    
    def exp_map(self, p, v):
        v_norm = torch.norm(v, 2, -1, True).clamp(min=1e-10)
        sqrt_c = np.sqrt(self.c)
        if isinstance(p, (int, float)) and p == 0:
            return torch.tanh(sqrt_c * v_norm) * v / (sqrt_c * v_norm)
        p_norm_sq = torch.sum(p*p, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        v_at_origin = torch.tanh(sqrt_c * lambda_p * v_norm / 2) * v / (sqrt_c * v_norm)
        return self.mobius_add(p, v_at_origin)

    def log_map(self, p, x):
        p_tensor = torch.zeros_like(x) if (isinstance(p, (int, float)) and p == 0) else p
        x_trans = self.mobius_add(-p_tensor, x)
        x_norm = torch.norm(x_trans, 2, -1, True).clamp(min=1e-10)
        p_norm_sq = torch.sum(p_tensor*p_tensor, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        scalar = (2 / (lambda_p * np.sqrt(self.c) * x_norm)) * torch.atanh((np.sqrt(self.c) * x_norm).clamp(max=1-1e-7))
        return scalar * x_trans

    def dist(self, x, y):
        sq_dist = torch.norm(x - y, 2, -1)**2
        nx_v, ny_v = 1 - self.c * torch.norm(x, 2, -1)**2, 1 - self.c * torch.norm(y, 2, -1)**2
        val = 1 + 2 * self.c * sq_dist / (nx_v.clamp(min=1e-15) * ny_v.clamp(min=1e-15))
        return (1 / np.sqrt(self.c)) * torch.acosh(val.clamp(min=1 + 1e-7))

    def project(self, x):
        norm = torch.norm(x, 2, -1, True)
        return torch.where(norm >= 0.99, x / norm * 0.99, x)

class RSGD(torch.optim.Optimizer):
    def __init__(self, params, lr, manifold): super().__init__(params, dict(lr=lr, manifold=manifold))
    def step(self):
        for group in self.param_groups:
            m, lr = group['manifold'], group['lr']
            for p in group['params']:
                if p.grad is None: continue
                rescale = ((1 - m.c * torch.sum(p.data**2, -1, True))**2 / 4)
                p.data = m.project(m.exp_map(p.data, -lr * p.grad.data * rescale))

# --- 2. RIEMANNIAN TRANSFORMER BLOCK ---
class PoincareTransformerBlock(nn.Module):
    def __init__(self, d, h, m):
        super().__init__()
        self.m, self.h, self.d_h = m, h, d // h
        self.q_lin, self.k_lin, self.v_lin = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
        
    def forward(self, x):
        b, s, d = x.shape
        q = self.m.exp_map(0, self.q_lin(self.ln1(x)).view(b, s, self.h, self.d_h)).transpose(1, 2)
        k = self.m.exp_map(0, self.k_lin(self.ln1(x)).view(b, s, self.h, self.d_h)).transpose(1, 2)
        v = self.v_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        
        # Hyperbolic Attention
        attn = torch.softmax(-self.m.dist(q.unsqueeze(3), k.unsqueeze(2)) / 1.0, -1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        
        # Geodesic Residual Update
        x = self.m.exp_map(0, self.m.log_map(0, x) + self.m.exp_map(0, out))
        x = self.m.exp_map(0, self.m.log_map(0, x) + self.ff(self.ln2(x)))
        return x

# --- 3. EXECUTION ---
def build_supply_chain(depth=3, branching=5):
    G = nx.balanced_tree(r=branching, h=depth)
    return nx.convert_node_labels_to_integers(G, label_attribute='name'), G

print("Constructing Multi-Tier Aerospace Supply Chain...")
mapping_G, raw_G = build_supply_chain()
vocab_size = len(mapping_G.nodes())
edge_index = torch.tensor(list(mapping_G.edges()), dtype=torch.long)

embed_dim, num_heads = 16, 4
manifold = PoincareManifold()
emb_layer = nn.Embedding(vocab_size, embed_dim)
model = PoincareTransformerBlock(embed_dim, num_heads, manifold)
optimizer = RSGD(list(emb_layer.parameters()) + list(model.parameters()), lr=0.5, manifold=manifold)

print(f"Training Riemannian Supply Chain Transformer (N={vocab_size})...")
for epoch in range(1501):
    optimizer.zero_grad()
    # Initial project to ball
    x_ball = manifold.exp_map(0, emb_layer(torch.arange(vocab_size)).unsqueeze(0))
    res = model(x_ball).squeeze(0)
    
    u, v = edge_index[:, 0], edge_index[:, 1]
    pos_dist = manifold.dist(res[u], res[v])
    neg_dist = manifold.dist(res[u], res[torch.randint(0, vocab_size, (u.size(0),))])
    
    # Loss favoring structural cohesion
    loss = pos_dist.mean() - 1.5 * torch.log(neg_dist.mean() + 1e-6)
    loss.backward()
    optimizer.step()
    
    if epoch % 500 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

# --- 4. EVALUATION ---
with torch.no_grad():
    final = model(manifold.exp_map(0, emb_layer(torch.arange(vocab_size)).unsqueeze(0))).squeeze(0)
    p_dists, g_dists = [], []
    for _ in range(1000):
        u, v = np.random.choice(vocab_size, 2, replace=False)
        g_dists.append(nx.shortest_path_length(mapping_G, u, v))
        p_dists.append(manifold.dist(final[u], final[v]).item())
        
    correlation = np.corrcoef(g_dists, p_dists)[0, 1]
    max_radius = torch.norm(final, p=2, dim=-1).max().item()

    print("\n" + "="*40)
    print(f"SUPPLY CHAIN RESULTS (RIEMANNIAN)")
    print("="*40)
    print(f"Correlation: {correlation:.4f}")
    print(f"Max Component Radius: {max_radius:.4f}")
    print("="*40)

Constructing Multi-Tier Aerospace Supply Chain...
Training Riemannian Supply Chain Transformer (N=156)...
Epoch 0 | Loss: 11.8931
Epoch 500 | Loss: 0.7005
Epoch 1000 | Loss: 0.5873
Epoch 1500 | Loss: 0.5649

SUPPLY CHAIN RESULTS (RIEMANNIAN)
Correlation: 0.0698
Max Component Radius: 0.8772


# Euclidian Embedded Transformer on Supply Chain Data.

In [15]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np

# --- 1. SUPPLY CHAIN GENERATOR ---
def build_supply_chain(depth=4, branching=4):
    """
    Simulates an Aerospace BOM:
    Level 0: Spacecraft (Root)
    Level 1: Subsystems (Propulsion, Avionics, Payload, EPS)
    Level 2: Assemblies (Thrusters, Batteries, Sensors)
    Level 3: Components (Valves, Cells, ICs)
    """
    G = nx.balanced_tree(r=branching, h=depth)
    return nx.convert_node_labels_to_integers(G, label_attribute='name'), G

# --- 2. EUCLIDEAN TRANSFORMER BLOCK ---
class EuclideanTransformerBlock(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.d_h = h, d // h
        self.q_lin, self.k_lin, self.v_lin = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
        
    def forward(self, x):
        b, s, d = x.shape
        q = self.q_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        k = self.k_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        v = self.v_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        
        # Euclidean Attention: Similarity based on L2 distance
        dist_sq = torch.cdist(q, k, p=2)**2
        attn = torch.softmax(-dist_sq / np.sqrt(self.d_h), dim=-1)
        
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        x = x + out
        x = x + self.ff(self.ln2(x))
        return x

# --- 3. EXECUTION ---
print("Generating Aerospace Supply Chain Structure...")
mapping_G, raw_G = build_supply_chain(depth=3, branching=5) # ~156 nodes
vocab_size = len(mapping_G.nodes())
edge_index = torch.tensor(list(mapping_G.edges()), dtype=torch.long)

embed_dim, num_heads = 16, 4
emb_layer = nn.Embedding(vocab_size, embed_dim)
model = EuclideanTransformerBlock(embed_dim, num_heads)
optimizer = torch.optim.Adam(list(emb_layer.parameters()) + list(model.parameters()), lr=0.01)

print(f"Training Euclidean Supply Chain Baseline (N={vocab_size})...")
for epoch in range(1501):
    optimizer.zero_grad()
    res = model(emb_layer(torch.arange(vocab_size)).unsqueeze(0)).squeeze(0)
    
    u, v = edge_index[:, 0], edge_index[:, 1]
    # Positive: nodes directly connected in the BOM
    pos_dist = torch.norm(res[u] - res[v], p=2, dim=-1)
    # Negative: random components that shouldn't be related
    neg_dist = torch.norm(res[u] - res[torch.randint(0, vocab_size, (u.size(0),))], p=2, dim=-1)
    
    loss = pos_dist.mean() - 1.5 * torch.log(neg_dist.mean() + 1e-6)
    loss.backward()
    optimizer.step()
    
    if epoch % 500 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

# --- 4. EVALUATION ---
with torch.no_grad():
    final = model(emb_layer(torch.arange(vocab_size)).unsqueeze(0)).squeeze(0)
    e_dists, g_dists = [], []
    for _ in range(1000):
        u, v = np.random.choice(vocab_size, 2, replace=False)
        g_dists.append(nx.shortest_path_length(mapping_G, u, v))
        e_dists.append(torch.norm(final[u] - final[v], p=2).item())
        
    correlation = np.corrcoef(g_dists, e_dists)[0, 1]
    max_norm = torch.norm(final, p=2, dim=-1).max().item()

    print("\n" + "="*40)
    print(f"SUPPLY CHAIN BASELINE (EUCLIDEAN)")
    print("="*40)
    print(f"Correlation: {correlation:.4f}")
    print(f"Max Part Displacement: {max_norm:.4f}")
    print("="*40)

Generating Aerospace Supply Chain Structure...
Training Euclidean Supply Chain Baseline (N=156)...
Epoch 0 | Loss: 3.1072
Epoch 500 | Loss: -4.1913
Epoch 1000 | Loss: -4.3588
Epoch 1500 | Loss: -4.4002

SUPPLY CHAIN BASELINE (EUCLIDEAN)
Correlation: 0.4471
Max Part Displacement: 96.6890


# Riemannian Embedded Transformer on Music Genres Data.

In [19]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
from nltk.corpus import wordnet as wn

# --- 1. MANIFOLDS & OPTIMIZERS ---
class PoincareManifold:
    def __init__(self, c=1.0): self.c = c
    def mobius_add(self, x, y):
        x2, y2, xy = torch.sum(x*x, -1, True), torch.sum(y*y, -1, True), torch.sum(x*y, -1, True)
        num = (1 + 2*self.c*xy + self.c*y2)*x + (1 - self.c*x2)*y
        denom = 1 + 2*self.c*xy + self.c**2 * x2*y2
        return num / torch.clamp(denom, min=1e-15)
    
    def exp_map(self, p, v):
        v_norm = torch.norm(v, 2, -1, True).clamp(min=1e-10)
        sqrt_c = np.sqrt(self.c)
        if isinstance(p, (int, float)) and p == 0:
            return torch.tanh(sqrt_c * v_norm) * v / (sqrt_c * v_norm)
        p_norm_sq = torch.sum(p*p, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        v_at_origin = torch.tanh(sqrt_c * lambda_p * v_norm / 2) * v / (sqrt_c * v_norm)
        return self.mobius_add(p, v_at_origin)

    def log_map(self, p, x):
        p_tensor = torch.zeros_like(x) if (isinstance(p, (int, float)) and p == 0) else p
        x_trans = self.mobius_add(-p_tensor, x)
        x_norm = torch.norm(x_trans, 2, -1, True).clamp(min=1e-10)
        p_norm_sq = torch.sum(p_tensor*p_tensor, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        scalar = (2 / (lambda_p * np.sqrt(self.c) * x_norm)) * torch.atanh((np.sqrt(self.c) * x_norm).clamp(max=1-1e-7))
        return scalar * x_trans

    def dist(self, x, y):
        sq_dist = torch.norm(x - y, 2, -1)**2
        nx_v, ny_v = 1 - self.c * torch.norm(x, 2, -1)**2, 1 - self.c * torch.norm(y, 2, -1)**2
        val = 1 + 2 * self.c * sq_dist / (nx_v.clamp(min=1e-15) * ny_v.clamp(min=1e-15))
        return (1 / np.sqrt(self.c)) * torch.acosh(val.clamp(min=1 + 1e-7))

    def project(self, x):
        norm = torch.norm(x, 2, -1, True)
        return torch.where(norm >= 0.99, x / norm * 0.99, x)

class RSGD(torch.optim.Optimizer):
    def __init__(self, params, lr, manifold): super().__init__(params, dict(lr=lr, manifold=manifold))
    def step(self):
        for group in self.param_groups:
            m, lr = group['manifold'], group['lr']
            for p in group['params']:
                if p.grad is None: continue
                rescale = ((1 - m.c * torch.sum(p.data**2, -1, True))**2 / 4)
                p.data = m.project(m.exp_map(p.data, -lr * p.grad.data * rescale))

# --- 2. TRANSFORMER BLOCKS ---
class PoincareTransformerBlock(nn.Module):
    def __init__(self, d, h, m):
        super().__init__()
        self.m, self.h, self.d_h = m, h, d // h
        self.q_lin, self.k_lin, self.v_lin = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
    def forward(self, x):
        b, s, d = x.shape
        q = self.m.exp_map(0, self.q_lin(self.ln1(x)).view(b, s, self.h, self.d_h)).transpose(1, 2)
        k = self.m.exp_map(0, self.k_lin(self.ln1(x)).view(b, s, self.h, self.d_h)).transpose(1, 2)
        v = self.v_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        attn = torch.softmax(-self.m.dist(q.unsqueeze(3), k.unsqueeze(2)) / 1.0, -1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        x = self.m.exp_map(0, self.m.log_map(0, x) + self.m.exp_map(0, out))
        x = self.m.exp_map(0, self.m.log_map(0, x) + self.ff(self.ln2(x)))
        return x

class EuclideanTransformerBlock(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.d_h = h, d // h
        self.q_lin, self.k_lin, self.v_lin = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
    def forward(self, x):
        b, s, d = x.shape
        q = self.q_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        k = self.k_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        v = self.v_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        dist_sq = torch.cdist(q, k, p=2)**2
        attn = torch.softmax(-dist_sq / np.sqrt(self.d_h), dim=-1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        x = x + out
        x = x + self.ff(self.ln2(x))
        return x

# --- 3. DATA PREPARATION ---
def build_music_taxonomy(root_name='music.n.01', max_nodes=300):
    G = nx.Graph()
    try:
        root = wn.synset(root_name)
    except:
        import nltk
        nltk.download('wordnet')
        root = wn.synset(root_name)
    nodes, visited = [root], {root}
    while nodes and len(G.nodes()) < max_nodes:
        current = nodes.pop(0)
        for child in current.hyponyms():
            if child not in visited:
                G.add_edge(current.name(), child.name())
                visited.add(child)
                nodes.append(child)
    if not nx.is_connected(G):
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()
    return nx.convert_node_labels_to_integers(G, label_attribute='name'), G

# --- 4. EXPERIMENT ENGINE ---
print("Crawling Music Genres...")
mapping_G, raw_G = build_music_taxonomy()
vocab_size = len(mapping_G.nodes())
edge_index = torch.tensor(list(mapping_G.edges()), dtype=torch.long)
embed_dim, num_heads = 16, 4

def run_experiment(mode='riemannian'):
    print(f"\n--- Training {mode.upper()} Music Model ---")
    emb = nn.Embedding(vocab_size, embed_dim)
    if mode == 'riemannian':
        m = PoincareManifold()
        model = PoincareTransformerBlock(embed_dim, num_heads, m)
        opt = RSGD(list(emb.parameters()) + list(model.parameters()), lr=0.5, manifold=m)
    else:
        model = EuclideanTransformerBlock(embed_dim, num_heads)
        opt = torch.optim.Adam(list(emb.parameters()) + list(model.parameters()), lr=0.01)

    for epoch in range(1501):
        opt.zero_grad()
        if mode == 'riemannian':
            x = m.exp_map(0, emb(torch.arange(vocab_size)).unsqueeze(0))
            res = model(x).squeeze(0)
            u, v = edge_index[:, 0], edge_index[:, 1]
            pos, neg = m.dist(res[u], res[v]), m.dist(res[u], res[torch.randint(0, vocab_size, (u.size(0),))])
            loss = pos.mean() - 2.0 * torch.log(neg.mean() + 1e-6)
        else:
            res = model(emb(torch.arange(vocab_size)).unsqueeze(0)).squeeze(0)
            u, v = edge_index[:, 0], edge_index[:, 1]
            pos, neg = torch.norm(res[u]-res[v], p=2, dim=-1), torch.norm(res[u]-res[torch.randint(0, vocab_size, (u.size(0),))], p=2, dim=-1)
            loss = pos.mean() - 2.0 * torch.log(neg.mean() + 1e-6)
        
        loss.backward()
        opt.step()
        if epoch % 500 == 0: print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

    with torch.no_grad():
        if mode == 'riemannian': final = model(m.exp_map(0, emb(torch.arange(vocab_size)).unsqueeze(0))).squeeze(0)
        else: final = model(emb(torch.arange(vocab_size)).unsqueeze(0)).squeeze(0)
        d_p, d_g = [], []
        for _ in range(1000):
            u, v = np.random.choice(vocab_size, 2, replace=False)
            d_g.append(nx.shortest_path_length(mapping_G, u, v))
            if mode == 'riemannian': d_p.append(m.dist(final[u], final[v]).item())
            else: d_p.append(torch.norm(final[u]-final[v], p=2).item())
        return np.corrcoef(d_g, d_p)[0, 1]

# Run results
r_corr = run_experiment('riemannian')
e_corr = run_experiment('euclidean')

print("\n" + "="*45)
print(f"MUSIC GENRE FINAL REPORT (N={vocab_size})")
print("="*45)
print(f"Riemannian Correlation (Poincare): {r_corr:.4f}")
print(f"Euclidean Correlation (Flat):      {e_corr:.4f}")
print(f"Performance Gap:                   {((r_corr-e_corr)/e_corr)*100:.1f}%")
print("="*45)

Crawling Music Genres...

--- Training RIEMANNIAN Music Model ---
Epoch 0 | Loss: 10.2660
Epoch 500 | Loss: 0.4441
Epoch 1000 | Loss: 0.3796
Epoch 1500 | Loss: 0.3094

--- Training EUCLIDEAN Music Model ---
Epoch 0 | Loss: 2.2133
Epoch 500 | Loss: -7.2463
Epoch 1000 | Loss: -7.2247
Epoch 1500 | Loss: -7.3267

MUSIC GENRE FINAL REPORT (N=307)
Riemannian Correlation (Poincare): 0.0634
Euclidean Correlation (Flat):      0.3867
Performance Gap:                   -83.6%


# Euclidian Embedded Transformer on Music Genres Data.

In [20]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
from nltk.corpus import wordnet as wn

# --- 1. EUCLIDEAN TRANSFORMER BLOCK ---
class EuclideanTransformerBlock(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.d_h = h, d // h
        self.q_lin = nn.Linear(d, d)
        self.k_lin = nn.Linear(d, d)
        self.v_lin = nn.Linear(d, d)
        self.ln1 = nn.LayerNorm(d)
        self.ln2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(
            nn.Linear(d, 4*d),
            nn.GELU(),
            nn.Linear(4*d, d)
        )
        
    def forward(self, x):
        b, s, d = x.shape
        # Project to Q, K, V
        q = self.q_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        k = self.k_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        v = self.v_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        
        # Standard Euclidean Dot-Product Attention
        # (Scaled by sqrt of head dimension)
        dist_sq = torch.cdist(q, k, p=2)**2
        attn = torch.softmax(-dist_sq / np.sqrt(self.d_h), dim=-1)
        
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        
        # Residual connections and Feed-Forward
        x = x + out
        x = x + self.ff(self.ln2(x))
        return x

# --- 2. DATA PREPARATION (WordNet Music) ---
def build_music_taxonomy(root_name='music.n.01', max_nodes=300):
    G = nx.Graph()
    try:
        root = wn.synset(root_name)
    except:
        import nltk
        nltk.download('wordnet')
        root = wn.synset(root_name)
        
    nodes, visited = [root], {root}
    while nodes and len(G.nodes()) < max_nodes:
        current = nodes.pop(0)
        for child in current.hyponyms():
            if child not in visited:
                G.add_edge(current.name(), child.name())
                visited.add(child)
                nodes.append(child)
                
    if not nx.is_connected(G):
        largest_cc = max(nx.connected_components(G), key=len)
        G = G.subgraph(largest_cc).copy()
    
    return nx.convert_node_labels_to_integers(G, label_attribute='name'), G

# --- 3. TRAINING CONFIG ---
print("Building Music Genre Baseline (Euclidean)...")
mapping_G, raw_G = build_music_taxonomy()
vocab_size = len(mapping_G.nodes())
edge_index = torch.tensor(list(mapping_G.edges()), dtype=torch.long)

embed_dim, num_heads = 16, 4
emb_layer = nn.Embedding(vocab_size, embed_dim)
model = EuclideanTransformerBlock(embed_dim, num_heads)
optimizer = torch.optim.Adam(list(emb_layer.parameters()) + list(model.parameters()), lr=0.01)

print(f"Training Euclidean Music Transformer (N={vocab_size})...")
for epoch in range(1501):
    optimizer.zero_grad()
    # Embeddings in R^n
    res = model(emb_layer(torch.arange(vocab_size)).unsqueeze(0)).squeeze(0)
    
    u, v = edge_index[:, 0], edge_index[:, 1]
    # Positive: Nodes that are parent/child
    pos_dist = torch.norm(res[u] - res[v], p=2, dim=-1)
    # Negative: Random genres (unlikely to be related)
    neg_dist = torch.norm(res[u] - res[torch.randint(0, vocab_size, (u.size(0),))], p=2, dim=-1)
    
    # Structural Loss
    loss = pos_dist.mean() - 2.0 * torch.log(neg_dist.mean() + 1e-6)
    loss.backward()
    optimizer.step()
    
    if epoch % 500 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

# --- 4. EVALUATION ---
with torch.no_grad():
    final = model(emb_layer(torch.arange(vocab_size)).unsqueeze(0)).squeeze(0)
    e_dists, g_dists = [], []
    for _ in range(1000):
        u, v = np.random.choice(vocab_size, 2, replace=False)
        g_dists.append(nx.shortest_path_length(mapping_G, u, v))
        e_dists.append(torch.norm(final[u] - final[v], p=2).item())
        
    correlation = np.corrcoef(g_dists, e_dists)[0, 1]
    max_norm = torch.norm(final, p=2, dim=-1).max().item()

    print("\n" + "="*40)
    print(f"MUSIC GENRE BASELINE (EUCLIDEAN)")
    print("="*40)
    print(f"Correlation: {correlation:.4f}")
    print(f"Max Embedding Norm: {max_norm:.4f}")
    print("="*40)

Building Music Genre Baseline (Euclidean)...
Training Euclidean Music Transformer (N=307)...
Epoch 0 | Loss: 2.0181
Epoch 500 | Loss: -6.1517
Epoch 1000 | Loss: -7.1137
Epoch 1500 | Loss: -7.2536

MUSIC GENRE BASELINE (EUCLIDEAN)
Correlation: -0.0613
Max Embedding Norm: 585.0166
